In [38]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

In [39]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [40]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [41]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [42]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

In [43]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

### COX assumption in Train data

In [44]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic      p  -log2(p)
MTV                       0.07   0.80      0.33
SUVpeak                   0.17   0.68      0.56
TLG                       0.38   0.54      0.90
age                       0.09   0.77      0.38
cavum_oris                0.00   0.99      0.02
charlson                  1.66   0.20      2.34
female                    2.67   0.10      3.29
histgrade_high            0.13   0.72      0.48
hpv_related              12.67 <0.005     11.39
hypopharynx               0.00   0.99      0.01
larynx                    0.00   0.98      0.03
oropharynx                0.00   0.98      0.02
pack_years                0.06   0.81      0.31
uicc8_III-IV              

In [45]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


In [46]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic    p  -log2(p)
MTV                       0.59 0.44      1.17
SUVpeak                   0.01 0.91      0.14
TLG                       0.00 0.96      0.06
age                       0.43 0.51      0.97
cavum_oris                0.41 0.52      0.94
charlson                  0.87 0.35      1.51
female                    2.39 0.12      3.04
histgrade_high            0.11 0.74      0.44
hpv_related               6.46 0.01      6.51
hypopharynx               0.04 0.84      0.25
larynx                    0.97 0.33      1.62
oropharynx                0.85 0.36      1.49
pack_years                0.00 0.99      0.01
uicc8_III-IV              0.00 0.95      0.07


In [47]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


###### penalizer values essentially result in same result 

## Test dataset: MAASTRO 

In [48]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [49]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [50]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [51]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [52]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [53]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event


In [54]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [55]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [56]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [57]:
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [58]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

In [59]:
# VIF dataframe 
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

vif_data

,feature,VIF
0,age,1.136852
1,female,1.197250
2,cavum_oris,7.993011
3,oropharynx,60.199385
4,hypopharynx,11.118710
5,larynx,14.183474
6,histgrade_high,1.150479
7,hpv_related,4.309385
8,charlson,1.302350
9,pack_years,1.647441


# Standardization

In [60]:
original_X = X.copy()

In [61]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']
### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
scaler = RobustScaler()  
X_numeric_std = scaler.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [68]:
# Standardize X_MAASTRO 
MAASTRO_new = X_MAASTRO.copy()
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [69]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = MAASTRO_new_std 

In [70]:
X_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,0.0,14.473272,7.934,86.228420
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,0.0,5.044678,1.656,7.040100
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,1.0,7.839043,14.502,83.569669
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,0.0,2.880631,2.440,5.567091
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,0.0,5.402006,3.668,16.150550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,0.0,9.290139,3.650,26.280140
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,1.0,7.172883,18.967,101.754834
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,0.0,13.873187,6.370,66.273201
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,1.0,7.507419,12.443,71.832443


In [71]:
X_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,-0.497635,1,0,1,0,0,1,0.0,0,-0.747766,0.0,0.600320,0.075008,0.231096
1,-0.473435,0,0,0,0,1,0,0.0,1,0.159774,0.0,-0.690705,-0.466877,-0.387041
2,-0.113739,0,1,0,0,0,1,0.0,1,0.610629,1.0,-0.308082,0.641923,0.210342
3,0.826312,0,0,0,0,1,0,0.0,1,0.494088,0.0,-0.987020,-0.399206,-0.398539
4,0.596634,0,0,0,0,1,0,0.0,1,1.007387,0.0,-0.641777,-0.293211,-0.315926
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,0.000000,0,0,1,0,0,1,1.0,0,-0.747766,0.0,-0.109388,-0.294765,-0.236855
135,0.671213,0,0,1,0,0,1,1.0,0,-0.747766,1.0,-0.399297,1.027319,0.352293
136,-0.235838,0,0,1,0,0,1,1.0,1,0.560275,0.0,0.518153,-0.059989,0.075327
137,0.421516,0,0,1,0,0,1,1.0,1,1.620942,1.0,-0.353490,0.464201,0.118722


In [72]:
MAASTRO_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623
1,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700
2,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342
3,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979
4,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782
95,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868
96,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274
97,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492


In [73]:
MAASTRO_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,-0.436476,0,0,1,0,0,1,1,1,-0.747766,0,0.732497,1.361702,1.615732
1,-0.436476,0,0,1,0,0,0,0,0,-0.085444,1,-0.172482,-0.121272,-0.153327
2,-0.436476,0,0,1,0,0,0,0,1,-0.549070,1,0.463784,0.062665,0.140609
3,0.045320,1,0,0,0,1,1,0,1,0.742458,1,-0.199405,0.072763,-0.076742
4,0.768012,0,0,1,0,0,1,1,1,1.206084,0,-0.041772,0.705364,0.398213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,0.446816,1,0,0,0,1,0,0,0,1.073619,1,2.909606,-0.082431,0.685886
95,0.205918,0,0,0,0,1,0,0,1,5.014436,1,0.404287,0.010099,0.098289
96,0.205918,0,0,1,0,0,1,1,1,-0.747766,1,-0.156713,0.812913,0.358846
97,-0.516775,0,0,1,0,0,1,1,0,-0.747766,0,0.560744,0.251694,0.363804


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [74]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 23:40:38,225] A new study created in memory with name: no-name-82791b90-4757-4adf-b3b6-28f4fa620c06


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7455357142857143


[I 2024-04-13 23:40:38,884] A new study created in memory with name: no-name-c2b11e71-9d6c-4db6-b768-d1d4fa4d48f4


Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:40:38,878] Trial 0 finished with value: 0.7580581707147598 and parameters: {}. Best is trial 0 with value: 0.7580581707147598.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7580581707147598], datetime_start=datetime.datetime(2024, 4, 13, 23, 40, 38, 311308), datetime_complete=datetime.datetime(2024, 4, 13, 23, 40, 38, 878696), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7580581707147598


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.18224790157453513
Fold 2 IBS: 0.17936185172891464
Fold 3 IBS: 0.1578324465205439
Fold 4 IBS: 0.14729407963908342
Fold 5 IBS: 0.2737533899229805
[I 2024-04-13 23:40:39,606] Trial 0 finished with value: 0.18809793387721152 and parameters: {}. Best is trial 0 with value: 0.18809793387721152.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18809793387721152], datetime_start=datetime.datetime(2024, 4, 13, 23, 40, 38, 943353), datetime_complete=datetime.datetime(2024, 4, 13, 23, 40, 39, 606298), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18809793387721152


In [75]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [76]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.758
train_ibs:  0.188


#### Test

In [77]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [78]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.585
IBS score: 0.288


In [79]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [80]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [81]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:40:40,755] A new study created in memory with name: no-name-ae8ca3ad-ddce-4148-9ad4-06f5d677c144


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-13 23:40:41,053] A new study created in memory with name: no-name-53e2c304-0a0d-4613-84a6-7cc8eed07a3e


Fold 1 C-index: 0.5606060606060606
Fold 2 C-index: 0.6763392857142857
Fold 3 C-index: 0.8333333333333334
Fold 4 C-index: 0.8502109704641351
Fold 5 C-index: 0.6197183098591549
[I 2024-04-13 23:40:41,047] Trial 0 finished with value: 0.708041591995394 and parameters: {}. Best is trial 0 with value: 0.708041591995394.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.708041591995394], datetime_start=datetime.datetime(2024, 4, 13, 23, 40, 40, 816017), datetime_complete=datetime.datetime(2024, 4, 13, 23, 40, 41, 47007), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.708041591995394


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397652057302996
Fold 2 IBS: 0.22157790931615212
Fold 3 IBS: 0.20453593725187927
Fold 4 IBS: 0.22473802829011877
Fold 5 IBS: 0.21812431395361034
[I 2024-04-13 23:40:41,444] Trial 0 finished with value: 0.21659054187695811 and parameters: {}. Best is trial 0 with value: 0.21659054187695811.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054187695811], datetime_start=datetime.datetime(2024, 4, 13, 23, 40, 41, 86848), datetime_complete=datetime.datetime(2024, 4, 13, 23, 40, 41, 444419), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054187695811


In [82]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [83]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.708
train_ibs:  0.217


#### Test

In [84]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [85]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.576


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [86]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [87]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:40:42,207] A new study created in memory with name: no-name-02f296cd-5db8-460d-8200-473dfcbc04fe


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.75
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848


[I 2024-04-13 23:40:43,034] A new study created in memory with name: no-name-87518a9e-8c7f-4a9b-b8d1-9bc4410ad949


Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:40:43,015] Trial 0 finished with value: 0.7590875381579397 and parameters: {}. Best is trial 0 with value: 0.7590875381579397.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7590875381579397], datetime_start=datetime.datetime(2024, 4, 13, 23, 40, 42, 233966), datetime_complete=datetime.datetime(2024, 4, 13, 23, 40, 43, 14966), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7590875381579397


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1825535042004205
Fold 2 IBS: 0.17773002786634848
Fold 3 IBS: 0.15619742941827763
Fold 4 IBS: 0.14485273175547853
Fold 5 IBS: 0.27153624181650643
[I 2024-04-13 23:40:43,991] Trial 0 finished with value: 0.18657398701140632 and parameters: {}. Best is trial 0 with value: 0.18657398701140632.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18657398701140632], datetime_start=datetime.datetime(2024, 4, 13, 23, 40, 43, 68993), datetime_complete=datetime.datetime(2024, 4, 13, 23, 40, 43, 991283), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18657398701140632


In [88]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [89]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.759
train_ibs:  0.187


#### Test

In [90]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [91]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.591


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.284


In [92]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [93]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:40:44,586] A new study created in memory with name: no-name-96bae138-8d2f-416c-b302-4b506e626927


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:40:45,441] Trial 0 finished with value: 0.7573018238722253 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7573018238722253.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:40:46,349] Trial 1 finished with value: 0.7573018238722253 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7573018238722253.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:40:47,251] Trial 2 finished with value: 0.7573018238722253 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:41:04,933] Trial 24 finished with value: 0.7581946810150825 and parameters: {'l1_ratio': 0.7805647035680947}. Best is trial 6 with value: 0.7590875381579397.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.75
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:41:05,646] Trial 25 finished with value: 0.7590875381579397 and parameters: {'l1_ratio': 0.9368557715647121}. Best is trial 6 with value: 0.7590875381579397.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:41:06,415] Trial 26 finished with value: 0.7217769320973337 and parameters: {'l1_ratio': 0.015423757551295547}. Best is tr

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:41:23,516] Trial 48 finished with value: 0.7581946810150825 and parameters: {'l1_ratio': 0.7768768486379835}. Best is trial 6 with value: 0.7590875381579397.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:41:24,333] Trial 49 finished with value: 0.7573018238722253 and parameters: {'l1_ratio': 0.8386679916089912}. Best is trial 6 with value: 0.7590875381579397.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.75
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:41:25,110] Trial 50 finished with value: 0.7590875381579397 and parameters: {'l1_ratio': 0.945704027694845}. Best is trial

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.1826077931498605
Fold 2 IBS: 0.17807036541987384
Fold 3 IBS: 0.15601223631782776
Fold 4 IBS: 0.14500598472390383
Fold 5 IBS: 0.27109809628569953
[I 2024-04-13 23:42:08,075] Trial 0 finished with value: 0.18655889517943308 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.18655889517943308.
Fold 1 IBS: 0.18280515480908568
Fold 2 IBS: 0.1787673748211404
Fold 3 IBS: 0.15594917775011002
Fold 4 IBS: 0.14527384594992757
Fold 5 IBS: 0.2706297038179428
[I 2024-04-13 23:42:09,141] Trial 1 finished with value: 0.1866850514296413 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.18655889517943308.
Fold 1 IBS: 0.18283959551120751
Fold 2 IBS: 0.1789034782187268
Fold 3 IBS: 0.15588280754342965
Fold 4 IBS: 0.14527339338927314
Fold 5 IBS: 0.27072271225104416
[I 2024-04-13 23:42:10,181] Trial 2 finished with value: 0.18672439738273625 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.18655889517943

Fold 1 IBS: 0.18256466387128126
Fold 2 IBS: 0.17775325710813536
Fold 3 IBS: 0.1561219585406359
Fold 4 IBS: 0.14491963112125805
Fold 5 IBS: 0.27139257235325287
[I 2024-04-13 23:42:35,675] Trial 25 finished with value: 0.1865504165989127 and parameters: {'l1_ratio': 0.891573855041803}. Best is trial 25 with value: 0.1865504165989127.
Fold 1 IBS: 0.18255235052419533
Fold 2 IBS: 0.17775294056676047
Fold 3 IBS: 0.15620082583229222
Fold 4 IBS: 0.1449556806578617
Fold 5 IBS: 0.27147732185473183
[I 2024-04-13 23:42:36,544] Trial 26 finished with value: 0.1865878238871683 and parameters: {'l1_ratio': 0.9167426588025347}. Best is trial 25 with value: 0.1865504165989127.
Fold 1 IBS: 0.1825905733374704
Fold 2 IBS: 0.1779132401664048
Fold 3 IBS: 0.15602721625065713
Fold 4 IBS: 0.14493665338556005
Fold 5 IBS: 0.27118662677252925
[I 2024-04-13 23:42:37,619] Trial 27 finished with value: 0.1865308619825243 and parameters: {'l1_ratio': 0.7760895212945972}. Best is trial 27 with value: 0.186530861982524

Fold 1 IBS: 0.18265222584925453
Fold 2 IBS: 0.17808199968168006
Fold 3 IBS: 0.156072645532077
Fold 4 IBS: 0.14510246717123726
Fold 5 IBS: 0.2710827089951062
[I 2024-04-13 23:43:01,950] Trial 50 finished with value: 0.18659840944587103 and parameters: {'l1_ratio': 0.644904156608236}. Best is trial 35 with value: 0.18650206391711513.
Fold 1 IBS: 0.18259512317265011
Fold 2 IBS: 0.17791587155937344
Fold 3 IBS: 0.15600954469632755
Fold 4 IBS: 0.1449229065237299
Fold 5 IBS: 0.271169892694163
[I 2024-04-13 23:43:02,852] Trial 51 finished with value: 0.1865226677292488 and parameters: {'l1_ratio': 0.7683040387171605}. Best is trial 35 with value: 0.18650206391711513.
Fold 1 IBS: 0.1825875510915538
Fold 2 IBS: 0.17791148348938352
Fold 3 IBS: 0.15607292551131957
Fold 4 IBS: 0.14494580024908724
Fold 5 IBS: 0.2712079998274259
[I 2024-04-13 23:43:03,875] Trial 52 finished with value: 0.186545152033754 and parameters: {'l1_ratio': 0.7813477291140531}. Best is trial 35 with value: 0.18650206391711513

Fold 1 IBS: 0.18263920187061838
Fold 2 IBS: 0.17793214078682978
Fold 3 IBS: 0.15607474994700646
Fold 4 IBS: 0.14504733776578016
Fold 5 IBS: 0.27121251621018866
[I 2024-04-13 23:43:30,109] Trial 75 finished with value: 0.1865811893160847 and parameters: {'l1_ratio': 0.7224568234068146}. Best is trial 35 with value: 0.18650206391711513.
Fold 1 IBS: 0.18258278766661878
Fold 2 IBS: 0.17790869369486004
Fold 3 IBS: 0.15609113733434268
Fold 4 IBS: 0.1449602564495217
Fold 5 IBS: 0.271241259157135
[I 2024-04-13 23:43:31,240] Trial 76 finished with value: 0.18655682686049563 and parameters: {'l1_ratio': 0.7897917878683155}. Best is trial 35 with value: 0.18650206391711513.
Fold 1 IBS: 0.18261361359526307
Fold 2 IBS: 0.17807420709146848
Fold 3 IBS: 0.15598991959182848
Fold 4 IBS: 0.14498845457625684
Fold 5 IBS: 0.2710575111288036
[I 2024-04-13 23:43:32,743] Trial 77 finished with value: 0.1865447411967241 and parameters: {'l1_ratio': 0.6876396731724712}. Best is trial 35 with value: 0.18650206391

In [94]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [95]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.759
train_ibs:  0.187


#### Test

In [96]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [97]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.980766121964777)

test_cindex : 0.591


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.7564077859143797)

test_ibs:  0.284


In [98]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [99]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 23:44:02,979] A new study created in memory with name: no-name-2a332b85-577b-4da3-81b0-b310285f748b


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.6339285714285714
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8544303797468354
Fold 5 C-index: 0.6408450704225352
[I 2024-04-13 23:44:34,402] Trial 0 finished with value: 0.7372744686944293 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7372744686944293.
Fold 1 C-index: 0.7012987012987013
Fold 2 C-index: 0.6875
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8607594936708861
Fold 5 C-index: 0.647887323943662
[I 2024-04-13 23:44:41,951] Trial 1 finished with value: 0.74027341750814 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_features': '

Fold 1 C-index: 0.70995670995671
Fold 2 C-index: 0.8392857142857143
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8755274261603375
Fold 5 C-index: 0.7230046948356808
[I 2024-04-13 23:46:02,873] Trial 16 finished with value: 0.7913196149300414 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 15, 'min_samples_leaf': 16, 'max_depth': 2, 'n_estimators': 62, 'oob_score': True, 'max_samples': 0.9622193970781673, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.07748782870655765, 'warm_start': True}. Best is trial 14 with value: 0.8012393138587386.
Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8839662447257384
Fold 5 C-index: 0.7417840375586855
[I 2024-04-13 23:46:04,853] Trial 17 finished with value: 0.7952544618554078 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 16, 'min_samples_leaf': 15, 'max_depth': 7, 'n_estimators': 127, 'oob_score': True, 'max_samples': 0.8172323984404596, 

Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.9071729957805907
Fold 5 C-index: 0.7981220657276995
[I 2024-04-13 23:47:01,295] Trial 31 finished with value: 0.8102976168343801 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 7, 'min_samples_leaf': 7, 'max_depth': 12, 'n_estimators': 317, 'oob_score': True, 'max_samples': 0.8965443067001324, 'max_features': None, 'min_weight_fraction_leaf': 0.04869831874272522, 'warm_start': True}. Best is trial 30 with value: 0.8141045941707695.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.9029535864978903
Fold 5 C-index: 0.7746478873239436
[I 2024-04-13 23:47:07,162] Trial 32 finished with value: 0.8044151254239035 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 6, 'min_samples_leaf': 7, 'max_depth': 13, 'n_estimators': 317, 'oob_score': True, 'max_samples': 0.7822231835771817,

...

Fold 1 IBS: 0.21398563901141657
Fold 2 IBS: 0.22134596740550985
Fold 3 IBS: 0.2048025990579434
Fold 4 IBS: 0.22461415055237854
Fold 5 IBS: 0.21851803782809134
[I 2024-04-14 00:02:29,848] Trial 32 finished with value: 0.21665327877106794 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 18, 'n_estimators': 451, 'oob_score': False, 'max_samples': 0.7396444271446992, 'max_features': None, 'min_weight_fraction_leaf': 0.4501495537267681}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.19337121417130673
Fold 2 IBS: 0.19511882700454072
Fold 3 IBS: 0.18642967451701625
Fold 4 IBS: 0.1895345717679471
Fold 5 IBS: 0.21856358775020004
[I 2024-04-14 00:02:55,209] Trial 33 finished with value: 0.19660357504220216 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 12, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 493, 'oob_score': False, 'max_samples': 0.9933327757837453, 'max_features': None, 'min_weight_fraction_leaf

Fold 1 IBS: 0.21398162469590376
Fold 2 IBS: 0.22136781149843104
Fold 3 IBS: 0.20480753036595514
Fold 4 IBS: 0.22461586717115936
Fold 5 IBS: 0.21858421480458282
[I 2024-04-14 00:07:33,919] Trial 48 finished with value: 0.21667140970720644 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 459, 'oob_score': False, 'max_samples': 0.8399357435042919, 'max_features': None, 'min_weight_fraction_leaf': 0.43949783817145754}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.18607291644289184
Fold 2 IBS: 0.1997166990883884
Fold 3 IBS: 0.17871196581310303
Fold 4 IBS: 0.1866755623669813
Fold 5 IBS: 0.21401906861191788
[I 2024-04-14 00:07:52,573] Trial 49 finished with value: 0.19303924246465648 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 6, 'min_samples_leaf': 20, 'max_depth': 6, 'n_estimators': 480, 'oob_score': False, 'max_samples': 0.9976678104279734, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.17214163753608527
Fold 2 IBS: 0.19020363580055616
Fold 3 IBS: 0.16550959046326688
Fold 4 IBS: 0.16526794959392554
Fold 5 IBS: 0.22833087784482511
[I 2024-04-14 00:31:00,844] Trial 64 finished with value: 0.1842907382477318 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 18, 'n_estimators': 359, 'oob_score': False, 'max_samples': 0.8662053800926933, 'max_features': None, 'min_weight_fraction_leaf': 0.39542258978805284}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.2139792007492792
Fold 2 IBS: 0.22131668306329724
Fold 3 IBS: 0.20481655835559517
Fold 4 IBS: 0.22465389902940344
Fold 5 IBS: 0.2186107338002615
[I 2024-04-14 00:31:19,191] Trial 65 finished with value: 0.21667541499956733 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 12, 'min_samples_leaf': 8, 'max_depth': 20, 'n_estimators': 370, 'oob_score': False, 'max_samples': 0.8126605473414451, 'max_features': None, 'min_weight_fraction_leaf'

Fold 1 IBS: 0.21398666839515476
Fold 2 IBS: 0.22133561292920703
Fold 3 IBS: 0.20476084552868884
Fold 4 IBS: 0.22460858228328426
Fold 5 IBS: 0.21850495327987215
[I 2024-04-14 00:35:38,837] Trial 80 finished with value: 0.2166393324832414 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 6, 'max_depth': 16, 'n_estimators': 442, 'oob_score': True, 'max_samples': 0.7543724309332009, 'max_features': None, 'min_weight_fraction_leaf': 0.43164085935118673}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.1718240887458899
Fold 2 IBS: 0.191437443203446
Fold 3 IBS: 0.16491345446124533
Fold 4 IBS: 0.16398408770898562
Fold 5 IBS: 0.22656733476064983
[I 2024-04-14 00:35:54,403] Trial 81 finished with value: 0.18374528177604332 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 8, 'max_depth': 18, 'n_estimators': 355, 'oob_score': False, 'max_samples': 0.8723363913158229, 'max_features': None, 'min_weight_fraction_leaf': 

Fold 1 IBS: 0.18206223269633873
Fold 2 IBS: 0.18820585180072097
Fold 3 IBS: 0.17369714014143833
Fold 4 IBS: 0.1751220212365198
Fold 5 IBS: 0.22437209861320465
[I 2024-04-14 00:40:12,030] Trial 96 finished with value: 0.1886918688976445 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 8, 'n_estimators': 448, 'oob_score': False, 'max_samples': 0.8430407084230132, 'max_features': None, 'min_weight_fraction_leaf': 0.41419337591128474}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.17261810934600746
Fold 2 IBS: 0.19291443628076307
Fold 3 IBS: 0.16469348295311864
Fold 4 IBS: 0.16293280131529528
Fold 5 IBS: 0.2265701418039605
[I 2024-04-14 00:40:30,651] Trial 97 finished with value: 0.183945794339829 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 499, 'oob_score': False, 'max_samples': 0.8784577990568653, 'max_features': None, 'min_weight_fraction_leaf': 0

In [100]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [101]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.826
train_ibs:  0.183


#### Test

In [102]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

In [103]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=19, max_features='log2', max_leaf_nodes=15,
                     max_samples=0.9995515317027949, min_samples_split=18,
                     min_weight_fraction_leaf=0.032175807966512454,
                     n_estimators=115, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.653


RandomSurvivalForest(max_depth=20, max_features=None, max_leaf_nodes=13,
                     max_samples=0.9248759152889626, min_samples_split=2,
                     min_weight_fraction_leaf=0.41894325026727597,
                     n_estimators=396, random_state=123)

test_ibs:  0.206


In [104]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [105]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [106]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 00:41:07,401] A new study created in memory with name: no-name-fe9cf7fd-4e66-498c-866c-28295458ccbd


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.8080357142857143
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6948356807511737
[I 2024-04-14 00:41:10,055] Trial 0 finished with value: 0.7836129950691764 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7836129950691764.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 00:41:18,196] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 3 C-index: 0.8284313725490197
Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6713615023474179
[I 2024-04-14 00:42:46,852] Trial 15 finished with value: 0.7684393821468035 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.7914452579440178.
Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.8013392857142857
Fold 3 C-index: 0.8333333333333334
Fold 4 C-index: 0.8248945147679325
Fold 5 C-index: 0.6267605633802817
[I 2024-04-14 00:42:50,990] Trial 16 finished with value: 0.77051229268592 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_l

Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7924107142857143
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.6572769953051644
[I 2024-04-14 00:43:54,776] Trial 30 finished with value: 0.7671596620665804 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 11, 'min_samples_leaf': 12, 'max_depth': 5, 'n_estimators': 459, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.8382504072008882, 'min_weight_fraction_leaf': 0.23846812660748434}. Best is trial 12 with value: 0.7914452579440178.
Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.8529411764705882
Fold 4 C-index: 0.8248945147679325
Fold 5 C-index: 0.6384976525821596
[I 2024-04-14 00:44:03,979] Trial 31 finished with value: 0.7712482704957379 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 5, 'min_samples_leaf': 16, 'max_depth': 13, 'n_estimators': 500, 'oob_score': True, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.16792251267623906
Fold 2 IBS: 0.2174484716230587
Fold 3 IBS: 0.16273781989489802
Fold 4 IBS: 0.1558940391227505
Fold 5 IBS: 0.23222422254692077
[I 2024-04-14 00:56:23,693] Trial 30 finished with value: 0.18724541317277338 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 6, 'min_samples_leaf': 2, 'max_depth': 12, 'n_estimators': 455, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.7293028765390641, 'min_weight_fraction_leaf': 0.08037045697723012}. Best is trial 23 with value: 0.1855727116611247.
Fold 1 IBS: 0.16646920043412164
Fold 2 IBS: 0.22144555868438975
Fold 3 IBS: 0.1612585234507463
Fold 4 IBS: 0.15276461377317607
Fold 5 IBS: 0.23760424924495455
[I 2024-04-14 00:56:30,912] Trial 31 finished with value: 0.18790842911747765 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 209, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.83

Fold 1 IBS: 0.16512803721910052
Fold 2 IBS: 0.21893230349393553
Fold 3 IBS: 0.1593836385256823
Fold 4 IBS: 0.15058205502042205
Fold 5 IBS: 0.23336241815587846
[I 2024-04-14 00:58:20,946] Trial 45 finished with value: 0.18547769048300378 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 3, 'max_depth': 12, 'n_estimators': 276, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6931500413934508, 'min_weight_fraction_leaf': 0.021457257714753555}. Best is trial 32 with value: 0.18528124104479504.
Fold 1 IBS: 0.1659546836323674
Fold 2 IBS: 0.2153197227930472
Fold 3 IBS: 0.16360867863772716
Fold 4 IBS: 0.15557836846786757
Fold 5 IBS: 0.233640633124344
[I 2024-04-14 00:58:29,409] Trial 46 finished with value: 0.18682041733107066 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 283, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0

Fold 1 IBS: 0.17342492293804432
Fold 2 IBS: 0.2110732537830341
Fold 3 IBS: 0.16831819527481992
Fold 4 IBS: 0.16873369129336893
Fold 5 IBS: 0.22655614890084055
[I 2024-04-14 01:01:09,607] Trial 60 finished with value: 0.18962124243802156 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 4, 'min_samples_leaf': 5, 'max_depth': 14, 'n_estimators': 439, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.5735294094022945, 'min_weight_fraction_leaf': 0.10077092974890593}. Best is trial 53 with value: 0.18257594357755064.
Fold 1 IBS: 0.16275315104094723
Fold 2 IBS: 0.22028966274589679
Fold 3 IBS: 0.16253743592400494
Fold 4 IBS: 0.15352895984062664
Fold 5 IBS: 0.23021227496706434
[I 2024-04-14 01:01:21,415] Trial 61 finished with value: 0.18586429690370798 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 4, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 412, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples':

Fold 1 IBS: 0.16278536069617075
Fold 2 IBS: 0.22061911570320727
Fold 3 IBS: 0.15948095174532076
Fold 4 IBS: 0.1500453215492157
Fold 5 IBS: 0.23108140998200039
[I 2024-04-14 01:04:15,333] Trial 75 finished with value: 0.18480243193518295 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 463, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.6487943567829829, 'min_weight_fraction_leaf': 0.011079149496446542}. Best is trial 53 with value: 0.18257594357755064.
Fold 1 IBS: 0.1649931201311961
Fold 2 IBS: 0.2191098546301487
Fold 3 IBS: 0.1604180424655896
Fold 4 IBS: 0.15133070246635846
Fold 5 IBS: 0.23270813963892614
[I 2024-04-14 01:04:32,435] Trial 76 finished with value: 0.18571197186644378 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 496, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0

Fold 1 IBS: 0.18162653052097805
Fold 2 IBS: 0.20612651784753333
Fold 3 IBS: 0.1771784733434652
Fold 4 IBS: 0.18941755917999278
Fold 5 IBS: 0.21614311489575638
[I 2024-04-14 01:07:10,179] Trial 90 finished with value: 0.19409843915754516 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 6, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 362, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.7635642567316503, 'min_weight_fraction_leaf': 0.280081595660318}. Best is trial 53 with value: 0.18257594357755064.
Fold 1 IBS: 0.16186089564069253
Fold 2 IBS: 0.21899259491528042
Fold 3 IBS: 0.15921485444738478
Fold 4 IBS: 0.148193042465612
Fold 5 IBS: 0.23190417972474658
[I 2024-04-14 01:07:22,010] Trial 91 finished with value: 0.18403311343874326 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 323, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.789

In [107]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [108]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.794
train_ibs:  0.183


#### Test

In [109]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [110]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=12, max_features=None, max_leaf_nodes=6,
                   max_samples=0.7030590704740037, min_samples_leaf=6,
                   min_samples_split=13,
                   min_weight_fraction_leaf=0.020016091327628317,
                   n_estimators=273, random_state=123, warm_start=True)

C-index score: 0.618


ExtraSurvivalTrees(max_depth=13, max_features='log2', max_leaf_nodes=6,
                   max_samples=0.7048480399273186, min_samples_leaf=1,
                   min_samples_split=12,
                   min_weight_fraction_leaf=0.0010291888027469595,
                   n_estimators=485, random_state=123, warm_start=True)

IBS: 0.215


In [111]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [112]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 01:08:58,194] A new study created in memory with name: no-name-51d6e374-5a0a-4860-94b4-751b2d541d3d


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:09:51,644] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:10:20,463] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:24:47,980] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7334874350516556.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:26:34,265] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:44:35,429] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7334874350516556.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 01:46:33,696] Trial 26 finished with value: 0.5421550134138162 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446,

...

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 05:07:03,312] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9027683411994925, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.13472529918097312, 'n_estimators': 381, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.36938177041618503, 'max_features': 'auto', 'min_impurity_decrease': 1.1016843774656315e-07, 'validation_fraction': 0.898549324711475, 'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 05:09:12,408] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.8324789538052517, 'learning_rate': 0.0985092209

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-14 05:34:13,521] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9104313304151431, 'learning_rate': 0.014815622348446418, 'dropout_rate': 0.1415896279444537, 'n_estimators': 149, 'criterion': 'squared_error', 'ccp_alpha': 0.7060910424624821, 'min_weight_fraction_leaf': 0.2796836959018326, 'max_features': 'auto', 'min_impurity_decrease': 5.127880425748818e-06, 'validation_fraction': 0.916501041728546, 'min_samples_split': 19, 'max_leaf_nodes': 16, 'min_samples_leaf': 18, 'max_depth': 5}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 05:36:35,769] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.802369464636257, 'learning_rate': 0.0041755238500

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 06:05:46,107] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8921532985730489, 'learning_rate': 0.021931279910218017, 'dropout_rate': 0.19284931428290142, 'n_estimators': 490, 'criterion': 'squared_error', 'ccp_alpha': 0.5905545096948588, 'min_weight_fraction_leaf': 0.07057220045251793, 'max_features': 'log2', 'min_impurity_decrease': 1.637559261311236e-05, 'validation_fraction': 0.8150762480319586, 'min_samples_split': 5, 'max_leaf_nodes': 13, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-14 06:08:23,261] Trial 78 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.7443996478445325, 'learning_rate': 0.086932868663

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 06:35:03,299] Trial 88 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9447987915285996, 'learning_rate': 0.014159070275535094, 'dropout_rate': 0.18102387516231752, 'n_estimators': 470, 'criterion': 'squared_error', 'ccp_alpha': 0.5935000311483954, 'min_weight_fraction_leaf': 0.2151508441811604, 'max_features': 'auto', 'min_impurity_decrease': 3.72786238165322e-07, 'validation_fraction': 0.898328514849679, 'min_samples_split': 18, 'max_leaf_nodes': 18, 'min_samples_leaf': 12, 'max_depth': 2}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21292314886383587
Fold 2 IBS: 0.2213935582997429
Fold 3 IBS: 0.20349492943164285
Fold 4 IBS: 0.2234443788901035
Fold 5 IBS: 0.2178743294294726
[I 2024-04-14 06:37:14,306] Trial 89 finished with value: 0.21582606898295956 and parameters: {'subsample': 0.9777744983689288, 'learning_rate': 0.010383571834622

Fold 3 IBS: 0.20237857642828805
Fold 4 IBS: 0.22105646622231043
Fold 5 IBS: 0.21749535106188492
[I 2024-04-14 06:54:50,186] Trial 99 finished with value: 0.21481876339222578 and parameters: {'subsample': 0.8569271041566342, 'learning_rate': 0.01377474153403631, 'dropout_rate': 0.2044520049398774, 'n_estimators': 378, 'criterion': 'squared_error', 'ccp_alpha': 0.004422201299274769, 'min_weight_fraction_leaf': 0.13940114558509004, 'max_features': 'auto', 'min_impurity_decrease': 0.00015837847703956763, 'validation_fraction': 0.9580200141092102, 'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 2}. Best is trial 99 with value: 0.21481876339222578.


* Best trial for IBS: 
 FrozenTrial(number=99, state=TrialState.COMPLETE, values=[0.21481876339222578], datetime_start=datetime.datetime(2024, 4, 14, 6, 53, 30, 434188), datetime_complete=datetime.datetime(2024, 4, 14, 6, 54, 50, 184938), params={'subsample': 0.8569271041566342, 'learning_rate': 0.013774741534

In [113]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [114]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.745
train_ibs:  0.215


#### Test

In [115]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [116]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.03063070291051248,
                                 criterion='squared_error',
                                 dropout_rate=0.2576884847115747,
                                 learning_rate=0.007359366951045268,
                                 max_depth=16, max_features=1,
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=1.3903906488794697e-07,
                                 min_samples_leaf=14, min_samples_split=20,
                                 min_weight_fraction_leaf=0.38180626899357545,
                                 n_estimators=96, random_state=123,
                                 subsample=0.8938290428827321,
                                 validation_fraction=0.9577535215137098)

C-index score: 0.603


GradientBoostingSurvivalAnalysis(ccp_alpha=0.004422201299274769,
                                 criterion='squared_error',
                                 dropout_rate=0.2044520049398774,
                                 learning_rate=0.01377474153403631, max_depth=2,
                                 max_features='auto', max_leaf_nodes=17,
                                 min_impurity_decrease=0.00015837847703956763,
                                 min_samples_leaf=13, min_samples_split=13,
                                 min_weight_fraction_leaf=0.13940114558509004,
                                 n_estimators=378, random_state=123,
                                 subsample=0.8569271041566342,
                                 validation_fraction=0.9580200141092102)

IBS: 0.22


In [117]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [118]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [119]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 06:55:08,165] A new study created in memory with name: no-name-495c7a63-0fe8-440c-b676-092766ee119f


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6294642857142857
Fold 3 C-index: 0.7401960784313726
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.5938967136150235
[I 2024-04-14 06:55:09,890] Trial 0 finished with value: 0.6977253889205401 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6977253889205401.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6339285714285714
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.5985915492957746
[I 2024-04-14 06:55:22,202] Trial 1 finished with value: 0.7034787818269984 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.7034787818269984.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.6339285714285714
Fold 3 C-index: 0.7647058823529411
Fold 

Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.6428571428571429
Fold 3 C-index: 0.7965686274509803
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.5868544600938967
[I 2024-04-14 06:57:21,610] Trial 19 finished with value: 0.7204404397458104 and parameters: {'subsample': 0.38793667853472646, 'dropout_rate': 0.19484380744875768, 'n_estimators': 431, 'learning_rate': 0.07766040270665416}. Best is trial 14 with value: 0.7368189957793195.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.65625
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.5915492957746479
[I 2024-04-14 06:57:32,204] Trial 20 finished with value: 0.7272820872866885 and parameters: {'subsample': 0.179916923106687, 'dropout_rate': 0.323440617921801, 'n_estimators': 420, 'learning_rate': 0.09611961349165574}. Best is trial 14 with value: 0.7368189957793195.
Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.6651785714285714
Fold 3 C-index: 0.8137254901960784
Fold 4 C-inde

Fold 1 C-index: 0.7445887445887446
Fold 2 C-index: 0.6741071428571429
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8396624472573839
Fold 5 C-index: 0.6291079812206573
[I 2024-04-14 07:00:38,462] Trial 38 finished with value: 0.7392579690671386 and parameters: {'subsample': 0.1004921614604077, 'dropout_rate': 0.15002677709758214, 'n_estimators': 487, 'learning_rate': 0.07111278775588788}. Best is trial 38 with value: 0.7392579690671386.
Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.6696428571428571
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.6150234741784038
[I 2024-04-14 07:00:49,694] Trial 39 finished with value: 0.7336093452113662 and parameters: {'subsample': 0.10773591528998765, 'dropout_rate': 0.5393862022989122, 'n_estimators': 483, 'learning_rate': 0.06892741183938003}. Best is trial 38 with value: 0.7392579690671386.
Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.6294642857142857
Fold 3 C-index: 0.7843137254901961

...

Fold 2 IBS: 0.250374844169466
Fold 3 IBS: 0.18845162054668713
Fold 4 IBS: 0.28514869201474047
Fold 5 IBS: 0.29666844679611676
[I 2024-04-14 07:09:08,325] Trial 19 finished with value: 0.23921335014137082 and parameters: {'subsample': 0.1754785939282794, 'dropout_rate': 0.5719526157391092, 'n_estimators': 131, 'learning_rate': 0.08478060933651231}. Best is trial 12 with value: 0.19582797901795795.
Fold 1 IBS: 0.19434502454492755
Fold 2 IBS: 0.1976087722642088
Fold 3 IBS: 0.1772542498943357
Fold 4 IBS: 0.20956005678120226
Fold 5 IBS: 0.2202624636892423
[I 2024-04-14 07:09:08,617] Trial 20 finished with value: 0.19980611343478333 and parameters: {'subsample': 0.361226391737001, 'dropout_rate': 0.6283466334631026, 'n_estimators': 54, 'learning_rate': 0.03263327531246306}. Best is trial 12 with value: 0.19582797901795795.
Fold 1 IBS: 0.18948383515874043
Fold 2 IBS: 0.19401132917288091
Fold 3 IBS: 0.17320106539943225
Fold 4 IBS: 0.19815111909120867
Fold 5 IBS: 0.21924291552130318
[I 2024-04-

Fold 2 IBS: 0.25964476688146426
Fold 3 IBS: 0.18481687137574168
Fold 4 IBS: 0.2961768422814036
Fold 5 IBS: 0.30148765224499835
[I 2024-04-14 07:09:19,527] Trial 38 finished with value: 0.24614971297906574 and parameters: {'subsample': 0.15910316352149195, 'dropout_rate': 0.13272164755980653, 'n_estimators': 182, 'learning_rate': 0.08686981422611556}. Best is trial 30 with value: 0.19256456291205953.
Fold 1 IBS: 0.1813402582893099
Fold 2 IBS: 0.2113066790483602
Fold 3 IBS: 0.16446340065812048
Fold 4 IBS: 0.20822632431489052
Fold 5 IBS: 0.2441580840500885
[I 2024-04-14 07:09:19,846] Trial 39 finished with value: 0.2018989492721539 and parameters: {'subsample': 0.4058461621623634, 'dropout_rate': 0.2877671506289493, 'n_estimators': 59, 'learning_rate': 0.06001166712014448}. Best is trial 30 with value: 0.19256456291205953.
Fold 1 IBS: 0.19118996884076425
Fold 2 IBS: 0.20718847221902204
Fold 3 IBS: 0.17518785992514377
Fold 4 IBS: 0.2115822426561496
Fold 5 IBS: 0.22955436013927263
[I 2024-0

Fold 3 IBS: 0.2551066308112034
Fold 4 IBS: 0.3158543420372131
Fold 5 IBS: 0.33439926479054727
[I 2024-04-14 07:09:27,993] Trial 57 finished with value: 0.29219425142301025 and parameters: {'subsample': 0.842164102951203, 'dropout_rate': 0.8246529447682569, 'n_estimators': 282, 'learning_rate': 0.08992583695045304}. Best is trial 51 with value: 0.19238085127787322.
Fold 1 IBS: 0.17469063190030448
Fold 2 IBS: 0.20638575044774324
Fold 3 IBS: 0.16866002115533307
Fold 4 IBS: 0.20823295748943074
Fold 5 IBS: 0.25213746425655137
[I 2024-04-14 07:09:28,348] Trial 58 finished with value: 0.20202136504987261 and parameters: {'subsample': 0.23552611614332428, 'dropout_rate': 0.7535089906852777, 'n_estimators': 69, 'learning_rate': 0.07090539770564733}. Best is trial 51 with value: 0.19238085127787322.
Fold 1 IBS: 0.17707132653006127
Fold 2 IBS: 0.19800059918930474
Fold 3 IBS: 0.1627930805552266
Fold 4 IBS: 0.1867302808015111
Fold 5 IBS: 0.23127888217137144
[I 2024-04-14 07:09:28,613] Trial 59 fini

Fold 1 IBS: 0.17853617145991055
Fold 2 IBS: 0.23588977410981765
Fold 3 IBS: 0.17081488781421916
Fold 4 IBS: 0.24289477474679425
Fold 5 IBS: 0.28857319241201684
[I 2024-04-14 07:09:44,317] Trial 77 finished with value: 0.22334176010855167 and parameters: {'subsample': 0.1001112939764896, 'dropout_rate': 0.5294003335641392, 'n_estimators': 143, 'learning_rate': 0.07314679264092239}. Best is trial 59 with value: 0.19117483384949502.
Fold 1 IBS: 0.1730450127166568
Fold 2 IBS: 0.22530630658873543
Fold 3 IBS: 0.17489061746957146
Fold 4 IBS: 0.23969480123538525
Fold 5 IBS: 0.2693371680473364
[I 2024-04-14 07:09:44,745] Trial 78 finished with value: 0.21645478121153708 and parameters: {'subsample': 0.23957179565357356, 'dropout_rate': 0.6270507767383827, 'n_estimators': 86, 'learning_rate': 0.07993068148087899}. Best is trial 59 with value: 0.19117483384949502.
Fold 1 IBS: 0.17256525203087672
Fold 2 IBS: 0.2519685332759536
Fold 3 IBS: 0.16589925362123678
Fold 4 IBS: 0.27773403389520596
Fold 5 

Fold 3 IBS: 0.16930034966284252
Fold 4 IBS: 0.19375258458530342
Fold 5 IBS: 0.22229018790927785
[I 2024-04-14 07:09:51,809] Trial 96 finished with value: 0.1923861160230203 and parameters: {'subsample': 0.10083687177713299, 'dropout_rate': 0.7805004063112455, 'n_estimators': 39, 'learning_rate': 0.06965774462731136}. Best is trial 59 with value: 0.19117483384949502.
Fold 1 IBS: 0.17683525286958884
Fold 2 IBS: 0.20779746239024227
Fold 3 IBS: 0.16051519951980997
Fold 4 IBS: 0.1820487332247056
Fold 5 IBS: 0.24023463248444205
[I 2024-04-14 07:09:52,146] Trial 97 finished with value: 0.19348625609775777 and parameters: {'subsample': 0.11856796934732222, 'dropout_rate': 0.9061069634081566, 'n_estimators': 66, 'learning_rate': 0.07017890475325032}. Best is trial 59 with value: 0.19117483384949502.
Fold 1 IBS: 0.21295907660299157
Fold 2 IBS: 0.2203245730751611
Fold 3 IBS: 0.2033231161410431
Fold 4 IBS: 0.22329249185147954
Fold 5 IBS: 0.2178623396577444
[I 2024-04-14 07:09:52,262] Trial 98 fini

In [120]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [121]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.742
train_ibs:  0.191


#### Test

In [122]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [123]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.1337822012086785,
                                              learning_rate=0.08418081739857108,
                                              n_estimators=453,
                                              random_state=123,
                                              subsample=0.12062136787385529)

C-index score: 0.541


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.5065802886056728,
                                              learning_rate=0.08322655251020356,
                                              n_estimators=45, random_state=123,
                                              subsample=0.10355122058653286)

IBS: 0.249


In [124]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [125]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.826,1.0
ExtraSurvivalTrees,0.794,2.0
CoxLasso,0.759,3.5
CoxElastic,0.759,3.5
CoxPH,0.758,5.0
GradientBoosting,0.745,6.0
ComponentwiseGradientBoosting,0.742,7.0
CoxRidge,0.708,8.0


In [126]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.183,1.5
ExtraSurvivalTrees,0.183,1.5
CoxLasso,0.187,3.5
CoxElastic,0.187,3.5
CoxPH,0.188,5.0
ComponentwiseGradientBoosting,0.191,6.0
GradientBoosting,0.215,7.0
CoxRidge,0.217,8.0


In [127]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.653,1.0
ExtraSurvivalTrees,0.618,2.0
GradientBoosting,0.603,3.0
CoxLasso,0.591,4.5
CoxElastic,0.591,4.5
CoxPH,0.585,6.0
CoxRidge,0.576,7.0
ComponentwiseGradientBoosting,0.541,8.0


In [128]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
Randomsurvivalforest,0.206,1.0
ExtraSurvivalTrees,0.215,2.0
GradientBoosting,0.220,3.0
CoxRidge,0.221,4.0
ComponentwiseGradientBoosting,0.249,5.0
CoxLasso,0.284,6.5
CoxElastic,0.284,6.5
CoxPH,0.288,8.0


In [129]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/os/robust/no_selection/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


# Modify the file names to match the desired format
modified_file_names = ['d1_os_robust_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [130]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
